# circuit-tracer native edge test: survives or collapses?

`circuit-tracer` builds an attribution graph on a transcoder replacement model and reports
which feature (and error) nodes influence a target logit. A graph edge is a candidate, not a
verdict. `saturn_pub.interop.circuit_tracer.native_edge_test` turns each selected edge into a
native intervention on the **real** Gemma-2-2B weights and lets the unchanged native consumer
decide whether the edge *survives* or *collapses*.

This notebook loads the committed, scrubbed verdict tables and re-derives every verdict
**in-process, offline** — no GPU, no model, no `circuit_tracer`, no torch. It reads only the
sealed `edge_verdicts.json` scalars and the frozen decision rule. Install with
`pip install -e .` from the checkout.

In [ ]:
import json
from pathlib import Path

# repo root (this notebook lives in notebooks/)
ROOT = Path.cwd()
ROOT = ROOT if (ROOT / 'experiments').exists() else ROOT.parent
BUNDLES = ROOT / 'experiments' / 'circuit_tracer' / 'bundles'
prompts = sorted(p.name for p in BUNDLES.iterdir() if (p / 'edge_verdicts.json').exists())
print('committed verdict tables:', prompts)

In [ ]:
def load_table(name):
    return json.loads((BUNDLES / name / 'edge_verdicts.json').read_text())

def show(name):
    t = load_table(name)
    s = t['summary']
    print(f"=== {name} ===")
    print(f"prompt: {t['prompt']!r}  -> target {t['target_text']!r}")
    print(f"clean top-1 is the answer: {t.get('clean_top1_is_answer')}")
    print(f"feature edges tested: {s['n_covered']}/{s['n_feature_edges']}  "
          f"surviving {s['n_surviving']}  collapsing {s['n_collapsing']}  "
          f"inconclusive {s['n_inconclusive']}")
    print(f"surviving fraction: {s['surviving_fraction']:.2f}   "
          f"error-node influence share: {s['error_node_influence_share']:.3f}")
    print(f"{'layer':>5} {'pos':>4} {'feat':>6} {'g_wt':>8} {'logitd':>8} {'flip':>5} "
          f"{'operator':>16}  verdict")
    for r in t['rows']:
        if r['node_kind'] != 'feature':
            continue
        gw = f"{r['graph_weight']:.3f}"
        ld = '' if r['native_logit_delta'] is None else f"{r['native_logit_delta']:.3f}"
        fl = '' if r['top_token_flip'] is None else ('yes' if r['top_token_flip'] else 'no')
        print(f"{r['layer']!s:>5} {r['position']!s:>4} {r['feature']!s:>6} {gw:>8} "
              f"{ld:>8} {fl:>5} {r['operator']:>16}  {r['verdict']}")
    err = [r for r in t['rows'] if r['node_kind'] == 'error'][0]
    print(f"error nodes (uncovered): {err['measurements']['n_error_nodes']} nodes, "
          f"{err['measurements']['error_node_influence_share']:.3f} of node influence")

for name in prompts:
    show(name)
    print()

## Fresh-process re-derivation (offline, stdlib)

`verify_edge_bundle` re-checks each bundle's SHA-256 and re-derives every covered verdict from
its sealed `native_logit_delta` scalar under the frozen `DecisionRule`, using the same
`trial.grade` the arbiter used. It imports no torch, no model, no `circuit_tracer`.

In [ ]:
from saturn_pub.interop.circuit_tracer import verify_edge_bundle

for name in prompts:
    result = verify_edge_bundle(BUNDLES / name)
    print(f"{name}: ok={result['ok']}  hash_ok={result['hash_ok']}  "
          f"fingerprint_ok={result['fingerprint_ok']}  verdict_ok={result['verdict_ok']}")
    assert result['ok'], result

## Reading

The verdict column is the native consumer's answer, not the graph's. `survives` means
removing the transcoder feature's decoder contribution on the real model dropped the answer's
log-prob by at least the rule's threshold; `collapses` means it moved the answer by almost
nothing despite the graph selecting it; the error-node share is influence the transcoder
features never expose. See [`docs/circuit-tracer.md`](../docs/circuit-tracer.md) and
[`experiments/circuit_tracer/README.md`](../experiments/circuit_tracer/README.md).